# Rico UI Drift Measurement v2 — Cross-Modal Validation

**Paper 2: Semantic Drift, Not Rank Collapse**

**Question:** Does semantic drift occur in vision (Rico UI), not just state space (Push-T)?

**Design:** Conditions match LeWM State experiment:
- Free encoder: Transformer → 3D projection (not 64D) + SIGReg
- Prescribed encoder: ShovJEPA with 3D prescribed axes (P, F, D)
- Track: covariance eigenvalues, R² transfer, effective rank every epoch
- 100 epochs, seed=42

**Why 3D + SIGReg:** v1 used 64D without SIGReg → collapse (condition number 10¹²).
R²=1.0 because dead dimensions don't move — stability through collapse.
To measure drift (not collapse), we need SIGReg to prevent collapse, and 3D to match
the prescribed dimensionality. Same conditions as LeWM → valid cross-modal comparison.

In [ ]:
!pip install datasets -q

from google.colab import drive
drive.mount('/content/drive')

import os, json, time, copy, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LinearRegression
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DRIVE_DIR = '/content/drive/MyDrive/rico_drift_v2'
os.makedirs(DRIVE_DIR, exist_ok=True)
DATA_DIR = '/content/shov_v5'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Drive: {DRIVE_DIR}')

## 1. Data (Rico UI)

In [ ]:
import datasets as dsets

if os.path.exists(DATA_DIR + '/done'):
    print('Data already cached')
else:
    os.makedirs(DATA_DIR, exist_ok=True)
    ds = dsets.load_dataset(
        path="shunk031/Rico",
        name="ui-screenshots-and-view-hierarchies",
        split="train[:500]",
    )
    print(f'Downloaded {len(ds)} samples')

    def extract_elements(node, elements):
        if not isinstance(node, dict):
            return
        bounds = node.get('bounds', None)
        if bounds is not None and isinstance(bounds, list) and len(bounds) > 0:
            if isinstance(bounds[0], list):
                bounds = bounds[0]
            if len(bounds) == 4:
                try:
                    x1, y1, x2, y2 = int(bounds[0]), int(bounds[1]), int(bounds[2]), int(bounds[3])
                    w, h = max(x2-x1, 1), max(y2-y1, 1)
                    if w > 1 and h > 1:
                        cx, cy = (x1+x2)/2/1440, (y1+y2)/2/2560
                        rw, rh = w/1440, h/2560
                        clickable = float(bool(node.get('clickable', False)))
                        scroll = node.get('scroll', {})
                        scrollable = 0.0
                        if isinstance(scroll, dict):
                            scrollable = float(bool(scroll.get('scrollable-horizontal', False) or
                                               scroll.get('scrollable-vertical', False)))
                        klass = str(node.get('klass', ''))
                        is_text = float('Text' in klass or 'Edit' in klass)
                        is_image = float('Image' in klass)
                        elements.append([cx, cy, rw, rh, clickable, scrollable, is_text, is_image])
                except (ValueError, TypeError):
                    pass
        children = node.get('children', [])
        if isinstance(children, list):
            for child in children:
                extract_elements(child, elements)

    N_ELEM = 16
    CF = 8
    padded = []
    labels = []

    for i in range(len(ds)):
        sample = ds[i]
        activity = sample.get('activity')
        if activity is None:
            continue
        elements = []
        root = activity.get('root', {})
        extract_elements(root, elements)
        top_children = activity.get('children', [])
        if isinstance(top_children, list):
            for child in top_children:
                extract_elements(child, elements)
        if len(elements) < 4:
            continue
        feats = elements[:N_ELEM]
        while len(feats) < N_ELEM:
            feats.append([0.0]*CF)
        padded.append(np.array(feats, dtype=np.float32))
        has_click = any(f[4] > 0 for f in elements)
        labels.append(1 if has_click else 0)
        if (i+1) % 100 == 0:
            print(f'  processed {i+1}, valid: {len(padded)}')

    X = np.stack(padded)
    y = np.array(labels)
    np.save(f'{DATA_DIR}/X.npy', X)
    np.save(f'{DATA_DIR}/y.npy', y)
    open(f'{DATA_DIR}/done', 'w').close()
    print(f'Saved: X={X.shape}, y={y.shape}, balance={y.mean():.2f}')

X = np.load(f'{DATA_DIR}/X.npy')
y = np.load(f'{DATA_DIR}/y.npy')
print(f'X={X.shape}, y={y.shape}, balance={y.mean():.2f}')

n = len(X)
n_val = max(1, n // 5)
perm = np.random.RandomState(42).permutation(n)
X_tr, y_tr = X[perm[n_val:]], y[perm[n_val:]]
X_vl, y_vl = X[perm[:n_val]], y[perm[:n_val]]

class RicoDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

tr_dl = DataLoader(RicoDataset(X_tr, y_tr), batch_size=32, shuffle=True, drop_last=True)
vl_dl = DataLoader(RicoDataset(X_vl, y_vl), batch_size=64)
print(f'Train: {len(X_tr)}, Val: {len(X_vl)}')

## 2. Architecture

**Key difference from v1:** Free encoder projects to 3D (not 64D) + SIGReg.
This matches LeWM State conditions: same output dimensionality, same regularization.

In [ ]:
# ============================================================
# SIGReg (from LeWM)
# ============================================================
class SIGReg(nn.Module):
    def __init__(self, knots=17, num_proj=256):
        super().__init__()
        self.num_proj = num_proj
        t = torch.linspace(0, 3, knots)
        dt = 3/(knots-1)
        w = torch.full((knots,), 2*dt); w[[0,-1]] = dt
        phi = torch.exp(-t.square()/2.0)
        self.register_buffer("t", t)
        self.register_buffer("phi", phi)
        self.register_buffer("weights", w*phi)
    def forward(self, proj):
        A = torch.randn(proj.size(-1), self.num_proj, device=proj.device)
        A = A.div_(A.norm(p=2, dim=0))
        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3)-self.phi).square() + x_t.sin().mean(-3).square()
        return ((err @ self.weights)*proj.size(-2)).mean()

# ============================================================
# Shared blocks
# ============================================================
class Attention(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.h=heads; self.d=dim//heads; self.s=self.d**-0.5
        self.qkv=nn.Linear(dim,dim*3); self.proj=nn.Linear(dim,dim)
    def forward(self, x):
        B,N,C=x.shape
        qkv=self.qkv(x).reshape(B,N,3,self.h,self.d).permute(2,0,3,1,4)
        q,k,v=qkv.unbind(0)
        out=(((q@k.transpose(-2,-1))*self.s).softmax(-1)@v).transpose(1,2).reshape(B,N,C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.n1=nn.LayerNorm(dim); self.attn=Attention(dim,heads)
        self.n2=nn.LayerNorm(dim)
        self.mlp=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(self, x): x=x+self.attn(self.n1(x)); return x+self.mlp(self.n2(x))

# ============================================================
# Free encoder: Transformer → 3D projection + SIGReg
# ============================================================
class FreeJEPA_3D(nn.Module):
    def __init__(self, cf=8, dim=64, depth=4, heads=4, pd=2, mr=0.5, ema=0.996, out_dim=3):
        super().__init__()
        self.mr=mr; self.ev=ema; self.dim=dim; self.out_dim=out_dim
        self.ce=nn.Linear(cf,dim)
        self.cls=nn.Parameter(torch.zeros(1,1,dim))
        self.pos=nn.Parameter(torch.zeros(1,17,dim))
        nn.init.trunc_normal_(self.pos,std=.02); nn.init.trunc_normal_(self.cls,std=.02)
        self.enc=nn.Sequential(*[Block(dim,heads) for _ in range(depth)])
        self.enorm=nn.LayerNorm(dim)
        self.proj_head=nn.Sequential(nn.Linear(dim,32),nn.GELU(),nn.Linear(32,out_dim))
        # EMA target
        self.tce=copy.deepcopy(self.ce); self.tenc=copy.deepcopy(self.enc)
        self.tnorm=copy.deepcopy(self.enorm); self.tproj=copy.deepcopy(self.proj_head)
        for p in list(self.tce.parameters())+list(self.tenc.parameters())+list(self.tnorm.parameters())+list(self.tproj.parameters()):
            p.requires_grad=False
        hd=dim//2
        self.pi=nn.Linear(out_dim,hd); self.mt=nn.Parameter(torch.zeros(1,1,hd))
        nn.init.trunc_normal_(self.mt,std=.02)
        self.pb=nn.Sequential(*[Block(hd,heads) for _ in range(pd)])
        self.po=nn.Linear(hd,out_dim)

    @torch.no_grad()
    def ema_update(self):
        for s,t in [(self.ce,self.tce),(self.enc,self.tenc),(self.enorm,self.tnorm),(self.proj_head,self.tproj)]:
            for a,b in zip(s.parameters(),t.parameters()):
                b.data=self.ev*b.data+(1-self.ev)*a.data

    def encode_3d(self, cells):
        B=cells.shape[0]
        x=self.ce(cells)
        x=torch.cat([self.cls.expand(B,-1,-1),x],1)+self.pos
        x=self.enorm(self.enc(x))
        return self.proj_head(x[:,0])  # CLS → 3D

    def forward(self, cells):
        B=cells.shape[0]; dev=cells.device; N=16; nm=int(N*self.mr)
        mask=torch.ones(B,N,dtype=torch.bool,device=dev)
        for b in range(B): mask[b,torch.randperm(N,device=dev)[:nm]]=False
        x=self.ce(cells)
        x=torch.cat([self.cls.expand(B,-1,-1),x],1)+self.pos
        fm=torch.cat([torch.ones(B,1,dtype=torch.bool,device=dev),mask],1)
        x=x*fm.unsqueeze(-1).float()
        x=self.enorm(self.enc(x)); ctx_3d=self.proj_head(x[:,1:])
        with torch.no_grad():
            t=self.tce(cells); t=torch.cat([self.cls.expand(B,-1,-1),t],1)+self.pos
            t=self.tnorm(self.tenc(t)); tc_3d=self.tproj(t[:,1:])
        p=self.pi(ctx_3d); mt=self.mt.expand(B,N,-1); m=mask.unsqueeze(-1).float()
        p=self.po(self.pb(p*m+mt*(1-m)))
        jl=torch.tensor(0.,device=dev)
        for b in range(B):
            mi=(~mask[b]).nonzero(as_tuple=True)[0]
            if len(mi)>0: jl=jl+F.mse_loss(p[b,mi],tc_3d[b,mi])
        return {'jl': jl/B, 'emb_3d': ctx_3d}

# ============================================================
# Prescribed encoder: ShovJEPA (3D prescribed axes)
# ============================================================
class ShovJEPA(nn.Module):
    def __init__(self, cf=8, dim=64, depth=4, heads=4, pd=2, mr=0.5, ema=0.996):
        super().__init__()
        self.mr=mr; self.ev=ema; self.dim=dim
        self.ce=nn.Linear(cf,dim)
        self.cls=nn.Parameter(torch.zeros(1,1,dim))
        self.pos=nn.Parameter(torch.zeros(1,17,dim))
        nn.init.trunc_normal_(self.pos,std=.02); nn.init.trunc_normal_(self.cls,std=.02)
        self.enc=nn.Sequential(*[Block(dim,heads) for _ in range(depth)])
        self.enorm=nn.LayerNorm(dim)
        self.tce=copy.deepcopy(self.ce); self.tenc=copy.deepcopy(self.enc); self.tnorm=copy.deepcopy(self.enorm)
        for p in list(self.tce.parameters())+list(self.tenc.parameters())+list(self.tnorm.parameters()):
            p.requires_grad=False
        hd=dim//2
        self.pi=nn.Linear(dim,hd); self.mt=nn.Parameter(torch.zeros(1,1,hd))
        nn.init.trunc_normal_(self.mt,std=.02)
        self.pb=nn.Sequential(*[Block(hd,heads) for _ in range(pd)]); self.po=nn.Linear(hd,dim)
        self.shov=nn.Sequential(nn.Linear(dim,32),nn.GELU(),nn.Linear(32,3),nn.Sigmoid())

    @torch.no_grad()
    def ema_update(self):
        for s,t in [(self.ce,self.tce),(self.enc,self.tenc),(self.enorm,self.tnorm)]:
            for a,b in zip(s.parameters(),t.parameters()):
                b.data=self.ev*b.data+(1-self.ev)*a.data

    def encode_3d(self, cells):
        B=cells.shape[0]
        x=self.ce(cells)
        x=torch.cat([self.cls.expand(B,-1,-1),x],1)+self.pos
        x=self.enorm(self.enc(x))
        return self.shov(x[:,0])

    def forward(self, cells, labels=None):
        B=cells.shape[0]; dev=cells.device; N=16; nm=int(N*self.mr)
        mask=torch.ones(B,N,dtype=torch.bool,device=dev)
        for b in range(B): mask[b,torch.randperm(N,device=dev)[:nm]]=False
        x=self.ce(cells)
        x=torch.cat([self.cls.expand(B,-1,-1),x],1)+self.pos
        fm=torch.cat([torch.ones(B,1,dtype=torch.bool,device=dev),mask],1)
        x=x*fm.unsqueeze(-1).float()
        x=self.enorm(self.enc(x)); cls_out=x[:,0]; ctx=x[:,1:]
        coords=self.shov(cls_out)
        sl=None
        if labels is not None:
            tgt=torch.zeros(B,3,device=dev)
            tgt[labels==0]=torch.tensor([0.,1.,1.],device=dev)
            tgt[labels==1]=torch.tensor([1.,0.,0.],device=dev)
            sl=F.mse_loss(coords,tgt)
        with torch.no_grad():
            t=self.tce(cells); t=torch.cat([self.cls.expand(B,-1,-1),t],1)+self.pos
            t=self.tnorm(self.tenc(t)); tc=t[:,1:]
        p=self.pi(ctx); mt=self.mt.expand(B,N,-1); m=mask.unsqueeze(-1).float()
        p=self.po(self.pb(p*m+mt*(1-m)))
        jl=torch.tensor(0.,device=dev)
        for b in range(B):
            mi=(~mask[b]).nonzero(as_tuple=True)[0]
            if len(mi)>0: jl=jl+F.mse_loss(p[b,mi],tc[b,mi])
        jl=jl/B
        return {'jl': jl, 'sl': sl, 'P': coords[:,0], 'F': coords[:,1], 'D': coords[:,2]}

print('Architecture defined: FreeJEPA_3D (3D + SIGReg) vs ShovJEPA (prescribed 3D)')

## 3. Drift Tracking

In [ ]:
def collect_embeddings_3d(model, dl):
    model.eval()
    all_emb = []
    with torch.no_grad():
        for X, y in dl:
            emb = model.encode_3d(X.to(device))
            all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb, 0)

def compute_cov_eigenvalues(emb):
    cov = np.cov(emb.T)
    return np.sort(np.linalg.eigvalsh(cov))[::-1]

def compute_r2_transfer(emb_prev, emb_curr):
    if emb_prev is None:
        return None
    reg = LinearRegression().fit(emb_prev, emb_curr)
    return reg.score(emb_prev, emb_curr)

def compute_effective_rank(eigenvalues):
    eigvals = np.array(eigenvalues)
    eigvals = eigvals[eigvals > 1e-12]
    p = eigvals / eigvals.sum()
    return np.exp(-np.sum(p * np.log(p + 1e-12)))

print('Drift tracking functions defined.')

## 4. Training with Drift Tracking

In [ ]:
EP = 100
LAM = 0.05  # SIGReg weight (same as LeWM)
torch.manual_seed(42)

# ============================================================
# FreeJEPA_3D (free encoder, 3D output, SIGReg)
# ============================================================
print('=== Training FreeJEPA_3D (free 3D + SIGReg) ===')
m_free = FreeJEPA_3D(cf=8, dim=64, depth=4, heads=4, out_dim=3).to(device)
sigreg = SIGReg(knots=17, num_proj=256).to(device)
opt_f = torch.optim.Adam(filter(lambda p: p.requires_grad, m_free.parameters()),
                          lr=1e-3, weight_decay=1e-4)
sch_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=EP)

hf = []
free_cov = []
free_r2 = []
free_rank = []
prev_emb_f = None

for ep in range(1, EP + 1):
    emb = collect_embeddings_3d(m_free, vl_dl)
    eigvals = compute_cov_eigenvalues(emb)
    r2 = compute_r2_transfer(prev_emb_f, emb)
    eff_rank = compute_effective_rank(eigvals)
    prev_emb_f = emb.copy()
    free_cov.append(eigvals.tolist())
    free_r2.append(r2)
    free_rank.append(eff_rank)

    m_free.train(); tl, ts = 0, 0
    for X, y in tr_dl:
        X = X.to(device)
        o = m_free(X)
        ls = sigreg(o['emb_3d'].reshape(-1, 3)) * LAM
        loss = o['jl'] + ls
        opt_f.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m_free.parameters(), 1.0)
        opt_f.step(); m_free.ema_update()
        tl += o['jl'].item(); ts += ls.item()
    sch_f.step()
    hf.append({'ep': ep, 'jl': tl/len(tr_dl), 'sig': ts/len(tr_dl)})
    r2s = f"{r2:.4f}" if r2 is not None else "N/A"
    if ep % 10 == 0 or ep == 1:
        print(f'  Ep {ep:3d} jl={tl/len(tr_dl):.4f} sig={ts/len(tr_dl):.4f} R²={r2s} rank={eff_rank:.2f}')

print(f'  Final: jl={hf[-1]["jl"]:.4f}')

# ============================================================
# ShovJEPA (prescribed 3D)
# ============================================================
print('\n=== Training ShovJEPA (prescribed 3D) ===')
torch.manual_seed(42)
m_shov = ShovJEPA(cf=8, dim=64, depth=4, heads=4).to(device)
opt_s = torch.optim.Adam(filter(lambda p: p.requires_grad, m_shov.parameters()),
                          lr=1e-3, weight_decay=1e-4)
sch_s = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s, T_max=EP)

hs_hist = []
shov_cov = []
shov_r2 = []
shov_rank = []
prev_emb_s = None
bs = 0

for ep in range(1, EP + 1):
    emb = collect_embeddings_3d(m_shov, vl_dl)
    eigvals = compute_cov_eigenvalues(emb)
    r2 = compute_r2_transfer(prev_emb_s, emb)
    eff_rank = compute_effective_rank(eigvals)
    prev_emb_s = emb.copy()
    shov_cov.append(eigvals.tolist())
    shov_r2.append(r2)
    shov_rank.append(eff_rank)

    m_shov.train(); tl, tc_, tt = 0, 0, 0
    for X, y in tr_dl:
        X, y = X.to(device), y.to(device)
        o = m_shov(X, y)
        loss = o['sl'] + 0.5 * o['jl']
        opt_s.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m_shov.parameters(), 1.0)
        opt_s.step(); m_shov.ema_update()
        tl += loss.item()
        tc_ += ((o['P'] > .5).long() == y).sum().item(); tt += y.size(0)
    m_shov.eval(); vc, vt = 0, 0
    with torch.no_grad():
        for X, y in vl_dl:
            X, y = X.to(device), y.to(device)
            o = m_shov(X, y)
            vc += ((o['P'] > .5).long() == y).sum().item(); vt += y.size(0)
    sch_s.step(); ta, va = tc_/tt, vc/vt
    hs_hist.append({'ep': ep, 'ta': ta, 'va': va, 'loss': tl/len(tr_dl)})
    if va > bs: bs = va
    r2s = f"{r2:.4f}" if r2 is not None else "N/A"
    if ep % 10 == 0 or ep == 1:
        print(f'  Ep {ep:3d} train={ta:.3f} val={va:.3f} R²={r2s} rank={eff_rank:.2f}')

print(f'  Best val acc: {bs:.3f}')

# Save to Drive
save_data = {
    'experiment': 'rico_drift_v2',
    'conditions': '3D output + SIGReg (matching LeWM)',
    'epochs': EP,
    'seed': 42,
    'free': {
        'history': hf,
        'cov_history': free_cov,
        'r2_history': free_r2,
        'rank_history': free_rank,
    },
    'shov': {
        'history': hs_hist,
        'cov_history': shov_cov,
        'r2_history': shov_r2,
        'rank_history': shov_rank,
        'best_val_acc': bs,
    }
}
with open(os.path.join(DRIVE_DIR, 'rico_drift_v2_results.json'), 'w') as f:
    json.dump(save_data, f, indent=2)
print(f'\nSaved to {DRIVE_DIR}/rico_drift_v2_results.json')

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# R² transfer
ax = axes[0, 0]
r2_f = [v if v is not None else np.nan for v in free_r2]
r2_s = [v if v is not None else np.nan for v in shov_r2]
ax.plot(r2_f, 'b-', label='Free 3D + SIGReg')
ax.plot(r2_s, 'k-', linewidth=2, label='ShovJEPA (prescribed 3D)')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('R²')
ax.set_title('R² Transfer Between Consecutive Epochs')
ax.legend(); ax.grid(True, alpha=0.3)

# Effective rank
ax = axes[0, 1]
ax.plot(free_rank, 'b-', label='Free 3D + SIGReg')
ax.plot(shov_rank, 'k-', linewidth=2, label='ShovJEPA (prescribed 3D)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Effective Rank')
ax.set_title('Effective Rank')
ax.legend(); ax.grid(True, alpha=0.3)

# Condition number
ax = axes[1, 0]
cond_f = [np.array(e)[0]/max(np.array(e)[-1],1e-12) for e in free_cov]
cond_s = [np.array(e)[0]/max(np.array(e)[-1],1e-12) for e in shov_cov]
ax.plot(cond_f, 'b-', label='Free 3D + SIGReg')
ax.plot(cond_s, 'k-', linewidth=2, label='ShovJEPA (prescribed 3D)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Condition Number')
ax.set_title('Covariance Condition Number')
ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3)

# Eigenvalue trajectories (free)
ax = axes[1, 1]
for dim_i in range(3):
    vals = [e[dim_i] for e in free_cov]
    ax.plot(vals, label=f'Free λ_{dim_i+1}')
for dim_i in range(3):
    vals = [e[dim_i] for e in shov_cov]
    ax.plot(vals, '--', label=f'Shov λ_{dim_i+1}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Eigenvalue')
ax.set_title('Eigenvalue Trajectories (3D)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.suptitle('Rico UI Drift v2 — Free 3D+SIGReg vs Prescribed 3D',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rico_drift_v2.png', dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(DRIVE_DIR, 'rico_drift_v2.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots.')

## 6. Summary

In [ ]:
print('\n' + '=' * 70)
print('RICO UI DRIFT v2 — SUMMARY')
print('=' * 70)

r2f = [v for v in free_r2 if v is not None]
r2s = [v for v in shov_r2 if v is not None]

print(f'\n  Free 3D + SIGReg:')
print(f'    R²(0→1) = {r2f[0]:.4f}')
print(f'    R² mean = {np.mean(r2f):.4f}')
print(f'    R² min  = {np.min(r2f):.4f}')
print(f'    Eff rank: {free_rank[0]:.2f} → {free_rank[-1]:.2f}')
print(f'    Cond num: {cond_f[0]:.1f} → {cond_f[-1]:.1f}')

print(f'\n  ShovJEPA (prescribed 3D):')
print(f'    R²(0→1) = {r2s[0]:.4f}')
print(f'    R² mean = {np.mean(r2s):.4f}')
print(f'    R² min  = {np.min(r2s):.4f}')
print(f'    Eff rank: {shov_rank[0]:.2f} → {shov_rank[-1]:.2f}')
print(f'    Cond num: {cond_s[0]:.1f} → {cond_s[-1]:.1f}')
print(f'    Best val acc: {bs:.3f}')

print(f'\n  Cross-modal comparison:')
print(f'    LeWM State (Push-T):  Free R²(0→1) = 0.78')
print(f'    Rico Vision (UI):     Free R²(0→1) = {r2f[0]:.4f}')

## 7. Download

In [ ]:
from google.colab import files

ZIP_DIR = '/content/rico_drift_v2_output'
os.makedirs(ZIP_DIR, exist_ok=True)
for fname in ['rico_drift_v2.png']:
    if os.path.exists(fname):
        shutil.copy(fname, ZIP_DIR)
drive_json = os.path.join(DRIVE_DIR, 'rico_drift_v2_results.json')
if os.path.exists(drive_json):
    shutil.copy(drive_json, ZIP_DIR)
shutil.make_archive('/content/rico_drift_v2_output', 'zip', ZIP_DIR)
files.download('/content/rico_drift_v2_output.zip')
print('Download started.')